# Customer Churn Prediction with a PyTorch ANN

This notebook trains a small feed-forward neural network to estimate the probability that a telecom customer will churn. It also exports the trained artifacts and verifies that the saved model can be used **without PyTorch** through `src/predict.py`.

**Model:** `45 -> 32 (ReLU) -> 16 (ReLU) -> 1 (logit)`  
**Loss:** `BCEWithLogitsLoss`  
**Optimizer:** Adam, learning rate `0.001`  
**Decision threshold:** selected on a validation split, never on the test set

> **Educational project.** The dataset contains historical customer records. The model is intended for learning and portfolio demonstration, not for production retention decisions.

### Training vs. inference dependencies

| Workflow | Dependencies |
| --- | --- |
| Notebook training | `requirements-train.txt` (includes PyTorch, plotting, Jupyter) |
| Production-style inference | `requirements.txt` (NumPy, pandas, scikit-learn, joblib; **no PyTorch**) |

The NumPy inference path loads `models/ann_weights.npz`, `models/preprocessor.pkl`, and `models/metadata.json`. The original `models/ann_customer_churn.pth` checkpoint is retained for reproducibility and optional parity checks.

## Contents

1. Setup and reproducibility
2. Exploratory data analysis
3. Data cleaning and preprocessing
4. ANN architecture and training
5. Validation-only threshold selection and final test evaluation
6. Artifact export and PyTorch-free inference
7. Verified results and limitations

> **A note on recorded outputs.** Training results depend on library versions and hardware. The training/evaluation cells below intentionally have no stored output, so stale results from another environment are not presented as current. Outputs in Section 6 were recorded by evaluating the committed artifacts in `models/`; they are reproduced by `scripts/evaluate.py` and `scripts/compare_backends.py`.

## 1. Setup and Reproducibility

In [ ]:
import copy
import json
import os
import random
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility: one seed for Python, NumPy, and PyTorch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(42)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# The notebook can be launched from the repository root or from notebooks/.
PROJECT_ROOT = Path(os.environ.get("CHURN_PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = Path(
    os.environ.get(
        "TELCO_DATA_PATH",
        PROJECT_ROOT / "data" / "raw" / "Telco-Customer-Churn.csv",
    )
)
MODEL_DIR = PROJECT_ROOT / "models"

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__} | PyTorch: {torch.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")

## 2. Exploratory Data Analysis

The dataset contains 7,043 customer records and 21 columns: 19 features, the `customerID` identifier, and the binary `Churn` target.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}.\n"
        "Download the IBM Telco Customer Churn CSV and place it at "
        "data/raw/Telco-Customer-Churn.csv, or set TELCO_DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")
df.head()

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print()
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values (before converting TotalCharges):")
print(df.isnull().sum())

print("\nChurn distribution:")
print(df["Churn"].value_counts())

print("\nChurn rate:")
print(df["Churn"].value_counts(normalize=True).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x="Churn", ax=axes[0])
axes[0].set_title("Churn distribution")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Customers")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly charges by churn")

sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[2])
axes[2].set_title("Tenure by churn")

plt.tight_layout()
plt.show()

## 3. Data Cleaning and Preprocessing

`TotalCharges` is stored as text and contains 11 blank values. Those rows are converted to missing values and removed. `customerID` is dropped because it carries no predictive information.

In [ ]:
blank_total_charges = (df["TotalCharges"].str.strip() == "").sum()
print("Blank TotalCharges values:", blank_total_charges)
print("TotalCharges dtype before conversion:", df["TotalCharges"].dtype)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values after conversion:", df["TotalCharges"].isnull().sum())

df = df.dropna().drop(columns=["customerID"]).reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("Remaining missing values:", int(df.isnull().sum().sum()))

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)
print(f"Features: {X.shape[1]} | Churn rate: {y.mean():.4f}")

### Stratified train/test split

The test set is held out before any model fitting. It is not used for training, early stopping, or threshold selection.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print(f"Training rows: {len(X_train_raw):,}")
print(f"Test rows: {len(X_test_raw):,}")
print(f"Training churn rate: {y_train.mean():.4f}")
print(f"Test churn rate: {y_test.mean():.4f}")

### Feature scaling and one-hot encoding

Numerical features are standardized and categorical features are one-hot encoded. The training portion is first split into a 4,500-row fitting fold and a 1,125-row validation fold. The preprocessor is fitted **only** on the fitting fold, then applied to validation and test rows. Validation statistics never influence scaling or category discovery.


In [ ]:
# Split raw training rows BEFORE fitting any transformer.  Fitting on
# X_train_raw would let validation rows influence scaler statistics and
# one-hot categories, contaminating early stopping and threshold selection.
X_fit_raw, X_val_raw, y_fit, y_val = train_test_split(
    X_train_raw,
    y_train,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train,
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

X_fit = preprocessor.fit_transform(X_fit_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

# ColumnTransformer can return a sparse matrix depending on the encoder output.
if hasattr(X_fit, "toarray"):
    X_fit = X_fit.toarray()
if hasattr(X_val, "toarray"):
    X_val = X_val.toarray()
if hasattr(X_test, "toarray"):
    X_test = X_test.toarray()

print("Processed training matrix:", X_fit.shape)
print("Processed validation matrix:", X_val.shape)
print("Processed test matrix:", X_test.shape)


## 4. ANN Architecture and Training

The network has two hidden layers. `BCEWithLogitsLoss` combines a sigmoid layer with binary cross-entropy, which is numerically more stable than applying sigmoid and then `BCELoss` separately.

In [ ]:
X_train_tensor = torch.tensor(X_fit, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(
    y_fit.to_numpy(), dtype=torch.float32
).reshape(-1, 1)
y_val_tensor = torch.tensor(
    y_val.to_numpy(), dtype=torch.float32
).reshape(-1, 1)
y_test_tensor = torch.tensor(
    y_test.to_numpy(), dtype=torch.float32
).reshape(-1, 1)

print("X_train:", tuple(X_train_tensor.shape))
print("X_val:  ", tuple(X_val_tensor.shape))
print("X_test: ", tuple(X_test_tensor.shape))
print("y_train:", tuple(y_train_tensor.shape))
print("y_val:  ", tuple(y_val_tensor.shape))
print("y_test: ", tuple(y_test_tensor.shape))


### Data loaders

The 4,500 fitting rows, 1,125 validation rows, and 1,407 test rows are already separated above. The validation rows are used for early stopping and threshold selection; the test rows remain untouched until Section 4's final evaluation.


In [ ]:
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=64,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_val_tensor, y_val_tensor),
    batch_size=64,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=64,
    shuffle=False,
)

print("Training samples:  ", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:      ", len(test_loader.dataset))


In [ ]:
class CustomerChurnANN(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, inputs):
        return self.network(inputs)


# Seed again immediately before construction so the initial weights are reproducible.
torch.manual_seed(42)
input_size = int(X_train_tensor.shape[1])
model = CustomerChurnANN(input_size)
print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

### Training loop with early stopping

Training runs for at most 50 epochs. Validation loss is monitored and the best state is restored; training stops after 5 epochs without improvement.

In [ ]:
epochs = 50
patience = 5
best_val_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, epochs + 1):
    model.train()
    running_train_loss = 0.0

    for inputs, targets in train_loader:
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * len(inputs)

    train_loss = running_train_loss / len(train_loader.dataset)

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            loss = criterion(model(inputs), targets)
            running_val_loss += loss.item() * len(inputs)

    val_loss = running_val_loss / len(val_loader.dataset)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} | "
            f"val loss {val_loss:.4f}"
        )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch} | best validation loss: {best_val_loss:.4f}")

### Validation-only threshold selection

The default threshold of 0.5 is not necessarily optimal for an imbalanced target. We search **0.20 to 0.70 in steps of 0.05** on the validation set and choose the threshold with the highest F1 score. The test set is not used in this search.

In [ ]:
model.eval()

val_probabilities = []
val_targets = []

with torch.no_grad():
    for inputs, targets in val_loader:
        probabilities = torch.sigmoid(model(inputs))
        val_probabilities.extend(probabilities.cpu().numpy().ravel())
        val_targets.extend(targets.cpu().numpy().ravel())

val_probabilities = np.asarray(val_probabilities)
val_targets = np.asarray(val_targets)

thresholds = np.round(np.arange(0.20, 0.71, 0.05), 2)
rows = []

for threshold in thresholds:
    predictions = (val_probabilities >= threshold).astype(int)
    rows.append(
        {
            "threshold": float(threshold),
            "precision": precision_score(val_targets, predictions, zero_division=0),
            "recall": recall_score(val_targets, predictions, zero_division=0),
            "f1": f1_score(val_targets, predictions, zero_division=0),
        }
    )

threshold_table = pd.DataFrame(rows)
print("Validation threshold search (0.20-0.70, step 0.05)")
print(
    threshold_table.to_string(
        index=False,
        formatters={
            "threshold": "{:.2f}".format,
            "precision": "{:.4f}".format,
            "recall": "{:.4f}".format,
            "f1": "{:.4f}".format,
        },
    )
)

best_index = int(threshold_table["f1"].idxmax())
best_threshold = float(threshold_table.loc[best_index, "threshold"])
best_val_f1 = float(threshold_table.loc[best_index, "f1"])

print(f"\nSelected threshold: {best_threshold:.2f}")
print(f"Validation F1: {best_val_f1:.4f}")

### Final held-out test evaluation

The selected threshold is applied once to the test set. ROC-AUC and average precision are threshold-independent and describe the ranking quality of the predicted probabilities.

In [ ]:
model.eval()

test_probabilities = []
test_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        probabilities = torch.sigmoid(model(inputs))
        test_probabilities.extend(probabilities.cpu().numpy().ravel())
        test_targets.extend(targets.cpu().numpy().ravel())

test_probabilities = np.asarray(test_probabilities)
test_targets = np.asarray(test_targets)
test_predictions = (test_probabilities >= best_threshold).astype(int)

print(f"Threshold: {best_threshold:.2f}")
print(f"Test samples: {len(test_targets)}")
print(f"Accuracy: {accuracy_score(test_targets, test_predictions):.4f}")
print(f"Precision: {precision_score(test_targets, test_predictions, zero_division=0):.4f}")
print(f"Recall: {recall_score(test_targets, test_predictions, zero_division=0):.4f}")
print(f"F1: {f1_score(test_targets, test_predictions, zero_division=0):.4f}")
print(f"ROC-AUC: {roc_auc_score(test_targets, test_probabilities):.4f}")
print(
    "Average precision: "
    f"{average_precision_score(test_targets, test_probabilities):.4f}"
)

print("\nClassification report:")
print(
    classification_report(
        test_targets,
        test_predictions,
        target_names=["No Churn", "Churn"],
        zero_division=0,
    )
)

print("Confusion matrix:")
print(confusion_matrix(test_targets, test_predictions))

In [ ]:
fpr, tpr, _ = roc_curve(test_targets, test_probabilities)
roc_auc = roc_auc_score(test_targets, test_probabilities)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ANN (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve - customer churn")
plt.legend()
plt.grid()
plt.show()

print("ROC-AUC:", round(roc_auc, 4))

In [ ]:
precision, recall, _ = precision_recall_curve(
    test_targets, test_probabilities
)
average_precision = average_precision_score(
    test_targets, test_probabilities
)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f"ANN (AP = {average_precision:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve - customer churn")
plt.legend()
plt.grid()
plt.show()

print("Average precision:", round(average_precision, 4))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    test_targets,
    test_predictions,
    display_labels=["No Churn", "Churn"],
    cmap="Blues",
)
plt.title("ANN - confusion matrix")
plt.show()

## 5. Exporting Inference Artifacts

The notebook writes four files to `models/`:

| File | Purpose |
| --- | --- |
| `ann_customer_churn.pth` | Original PyTorch `state_dict` (training/reproducibility) |
| `ann_weights.npz` | Identical weights exported as NumPy arrays (inference) |
| `preprocessor.pkl` | Fitted scikit-learn preprocessing pipeline |
| `metadata.json` | Architecture, feature columns, threshold, and training metadata |

The `.npz` keys intentionally match the PyTorch `state_dict` names so the two representations can be compared directly.

> **Re-running the notebook retrains the model.** The training cells rewrite the files in `models/`, and a fresh run in a different environment can produce slightly different weights and scores. The metrics in Section 7 describe the committed artifacts, not an arbitrary future run.

> **Artifact provenance.** The committed artifacts in `models/` were trained before the split-order correction above was applied (the earlier version fitted the preprocessor on all 5,625 training rows). They are left unchanged here so the reported metrics continue to describe the exact committed files. A fresh run of this notebook will fit preprocessing on the 4,500-row fitting fold only and will produce a new, internally consistent set of artifacts.


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

preprocessor_path = MODEL_DIR / "preprocessor.pkl"
checkpoint_path = MODEL_DIR / "ann_customer_churn.pth"
weights_path = MODEL_DIR / "ann_weights.npz"
metadata_path = MODEL_DIR / "metadata.json"

joblib.dump(preprocessor, preprocessor_path)
torch.save(model.state_dict(), checkpoint_path)

metadata = {
    "input_size": input_size,
    "architecture": [input_size, 32, 16, 1],
    "hidden_activation": "ReLU",
    "output_activation": "Sigmoid",
    "decision_rule": "churn_probability >= best_threshold",
    "best_threshold": best_threshold,
    "best_epoch": best_epoch,
    "numerical_columns": numerical_cols,
    "categorical_columns": categorical_cols,
}
metadata_path.write_text(
    json.dumps(metadata, indent=4) + "\n",
    encoding="utf-8",
)

state_dict = model.state_dict()
np.savez(
    weights_path,
    **{
        key: value.detach().cpu().numpy().astype(np.float32)
        for key, value in state_dict.items()
    },
)

print("Saved artifacts:")
for path in (checkpoint_path, weights_path, preprocessor_path, metadata_path):
    print(f"  {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size:,} bytes)")

In [ ]:
# Verify that the exported NumPy weights round-trip exactly.
with np.load(weights_path, allow_pickle=False) as exported:
    for key, tensor in model.state_dict().items():
        expected = tensor.detach().cpu().numpy().astype(np.float32)
        if not np.array_equal(exported[key], expected):
            raise RuntimeError(f"Weight mismatch after export: {key}")

print(f"Verified {len(state_dict)} weight tensors in ann_weights.npz.")

## 6. PyTorch-Free Inference

`src/predict.py` performs the forward pass with NumPy:

```
hidden_1 = ReLU(X @ W1.T + b1)
hidden_2 = ReLU(hidden_1 @ W2.T + b2)
logit    = hidden_2 @ W3.T + b3
probability = sigmoid(logit)
```

It validates the input columns, rejects missing/non-numeric values, applies the saved preprocessor, and uses `best_threshold` from `metadata.json`. It never imports PyTorch.

The first five rows of the raw dataset are used below only as a demonstration. No expected probability is hard-coded.

In [ ]:
# Make the repository's src/ package importable from either launch directory.
sys.path.insert(0, str(PROJECT_ROOT))

from src.predict import ChurnPredictor

numpy_predictor = ChurnPredictor(MODEL_DIR)
raw_customers = pd.read_csv(DATA_PATH)

print(f"Threshold: {numpy_predictor.threshold:.2f}")
print(f"Architecture: {numpy_predictor.architecture}")
print()

for customer_number, result in enumerate(
    numpy_predictor.predict_churn(raw_customers.iloc[:5]), start=1
):
    print(
        f"Customer {customer_number}: "
        f"probability={result['churn_probability']:.4f} "
        f"prediction={result['prediction']}"
    )

Threshold: 0.30
Architecture: [45, 32, 16, 1]

Customer 1: probability=0.5180 prediction=Churn
Customer 2: probability=0.0455 prediction=No Churn
Customer 3: probability=0.2525 prediction=No Churn
Customer 4: probability=0.0371 prediction=No Churn
Customer 5: probability=0.6859 prediction=Churn


### Reproducing the committed test metrics

The cell below rebuilds the same stratified train/test split from the raw data and evaluates the NumPy predictor against the held-out test rows. The output shown was recorded against the artifacts committed in `models/`; `scripts/evaluate.py` produces the same numbers.

In [ ]:
_, X_test_check, _, y_test_check = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

numpy_probabilities = numpy_predictor.predict_proba(X_test_check)
numpy_predictions = (
    numpy_probabilities >= numpy_predictor.threshold
).astype(int)

print("NumPy-only artifact verification")
print("-" * 40)
print(f"Threshold: {numpy_predictor.threshold:.2f}")
print(f"Test samples: {len(y_test_check)}")
print(f"Accuracy: {accuracy_score(y_test_check, numpy_predictions):.4f}")
print(
    "Precision: "
    f"{precision_score(y_test_check, numpy_predictions, zero_division=0):.4f}"
)
print(
    "Recall: "
    f"{recall_score(y_test_check, numpy_predictions, zero_division=0):.4f}"
)
print(f"F1: {f1_score(y_test_check, numpy_predictions, zero_division=0):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test_check, numpy_probabilities):.4f}")
print(
    "Average precision: "
    f"{average_precision_score(y_test_check, numpy_probabilities):.4f}"
)
print("Confusion matrix:")
print(confusion_matrix(y_test_check, numpy_predictions))

NumPy-only artifact verification
----------------------------------------
Threshold: 0.30
Test samples: 1407
Accuracy: 0.7619
Precision: 0.5377
Recall: 0.7433
F1: 0.6240
ROC-AUC: 0.8343
Average precision: 0.6245
Confusion matrix:
[[794 239]
 [ 96 278]]


### Optional PyTorch parity check

PyTorch is not required for inference. This optional check loads the original `.pth` checkpoint, confirms that every exported NumPy tensor is identical, and compares probabilities on the test set. It can also be run with:

```bash
python scripts/compare_backends.py
```

In [ ]:
from src.predict import WEIGHT_KEYS

state_dict = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)
torch_model = CustomerChurnANN(input_size)
torch_model.load_state_dict(state_dict)
torch_model.eval()

for key in WEIGHT_KEYS:
    torch_weight = state_dict[key].detach().cpu().numpy()
    if not np.array_equal(numpy_predictor.weights[key], torch_weight):
        raise RuntimeError(f"NumPy/PyTorch weight mismatch: {key}")

print(f"Weight keys exported: {len(WEIGHT_KEYS)}")
print("Exported weights match PyTorch exactly.")

first_five_matrix = numpy_predictor.preprocess(raw_customers.iloc[:5])
with torch.no_grad():
    torch_first_five = torch.sigmoid(
        torch_model(torch.tensor(first_five_matrix, dtype=torch.float32))
    ).cpu().numpy().ravel()

numpy_first_five = numpy_predictor.predict_proba(raw_customers.iloc[:5])

print(f"Customer 1 probability (PyTorch): {torch_first_five[0]:.4f}")
print(f"Customer 1 probability (NumPy):   {numpy_first_five[0]:.4f}")

test_matrix = numpy_predictor.preprocess(X_test_check)
with torch.no_grad():
    torch_test_probabilities = torch.sigmoid(
        torch_model(torch.tensor(test_matrix, dtype=torch.float32))
    ).cpu().numpy().ravel()

numpy_test_probabilities = numpy_predictor.predict_proba(X_test_check)
maximum_difference = np.max(
    np.abs(torch_test_probabilities - numpy_test_probabilities)
)

torch_predictions = (
    torch_test_probabilities >= numpy_predictor.threshold
).astype(int)
numpy_test_predictions = (
    numpy_test_probabilities >= numpy_predictor.threshold
).astype(int)
disagreements = int(np.sum(torch_predictions != numpy_test_predictions))

print(f"Max |PyTorch - NumPy| on test set: {maximum_difference:.3e}")
print(
    "Prediction disagreements at threshold "
    f"{numpy_predictor.threshold:.2f}: "
    f"{disagreements} / {len(numpy_test_predictions)}"
)

Weight keys exported: 6
Exported weights match PyTorch exactly.
Customer 1 probability (PyTorch): 0.5180
Customer 1 probability (NumPy):   0.5180
Max |PyTorch - NumPy| on test set: 1.192e-07
Prediction disagreements at threshold 0.30: 0 / 1407


## 7. Verified Results

These are the held-out test metrics for the committed artifacts, reproduced by `scripts/evaluate.py` and `docs/metrics.json`.

| Metric | Value |
| --- | ---: |
| Decision threshold | 0.30 |
| Test samples | 1,407 |
| Positive test samples | 374 |
| Accuracy | 0.7619 |
| Precision | 0.5377 |
| Recall | 0.7433 |
| F1 score | 0.6240 |
| ROC-AUC | 0.8343 |
| Average precision | 0.6245 |

**Confusion matrix** (rows = actual, columns = predicted):

|  | No Churn | Churn |
| --- | ---: | ---: |
| **No Churn** | 794 | 239 |
| **Churn** | 96 | 278 |

The threshold of 0.30 was chosen on the 1,125-row validation split, where it achieved the best validation F1 (0.6257). The test set was only used for the final evaluation above.

### A note on historical outputs

Older notebook output or screenshots from a different training run may show different probabilities. The committed artifacts are the source of truth. The verification cells above evaluate those artifacts directly; the first customer currently scores `0.5180` (`Churn`).

### Limitations

- The dataset is small and comes from a single historical snapshot.
- Metrics come from one stratified split; no cross-validation or confidence intervals are reported.
- The network is small and uncalibrated. Probabilities should not be interpreted as precise real-world risk.
- The threshold maximizes F1, which may not match the business cost of false positives versus false negatives.
- This project is educational and is not a production retention system.

## 8. Reproducing the Project

```bash
# Training / notebook dependencies (includes PyTorch)
pip install -r requirements-train.txt

# Inference-only dependencies (no PyTorch)
pip install -r requirements.txt

# Reproduce the held-out test metrics from the committed artifacts
python scripts/evaluate.py

# Optional NumPy vs. PyTorch parity check
python scripts/compare_backends.py

# Test suite
pytest -q
```

The raw Kaggle CSV is not committed. Place it at `data/raw/Telco-Customer-Churn.csv` or set `TELCO_DATA_PATH` to its location. See `README.md` for the full project overview.